# 03 — Corrected synthetic data and held-out evaluation

This notebook replaces the synthetic-data and evaluation steps of `02_model_training.ipynb`.

**What it fixes**
1. **Synthetic barangay shares.** Each barangay's share is now measured against **all Marikina cases**
   (the raw DOH files hold every case in the city), not only the three target barangays.
   Before, the whole city's yearly total was split among just three barangays — about 6× too high.
2. **Climate lag for synthetic years.** The 2017–2021 climate is now shifted 4 weeks, the same as the
   recorded data (`rainfall_lag` = rainfall 4 weeks earlier).
3. **Honest evaluation.** Models are trained on data **before 2025** and tested only on
   **recorded 2025–2026 weeks they never saw**, and compared against simple baselines.

Run the cells in order. They read and write files in `../data` and `../models`, like notebook 02.

In [1]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, recall_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    def make_xgb():
        return xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=42, verbosity=0)
except ImportError:  # only used if xgboost is missing
    from sklearn.ensemble import HistGradientBoostingRegressor
    def make_xgb():
        return HistGradientBoostingRegressor(max_depth=6, learning_rate=0.1, random_state=42)
    print('xgboost not installed — using a stand-in. Run: pip install xgboost')

warnings.filterwarnings('ignore')

DATA_DIR  = Path('../data')
MODEL_DIR = Path('../models')
TARGET_BARANGAYS = ['Nangka', 'Tumana', 'Malanday']

# Train on everything before this date; test on recorded weeks from this date on
TEST_START = pd.Timestamp('2025-01-01')

print('Libraries loaded.')

Libraries loaded.


## Step 1 — Barangay shares of all Marikina cases (2022–2026)

In [2]:
df_real = pd.read_csv(DATA_DIR / 'processed_dataset.csv')
df_real['week_start'] = pd.to_datetime(df_real['week_start'])

# Every row in the raw DOH files is one Marikina case (all barangays).
# Counting rows gives the city-wide total for 2022–2026.
RAW_FILES = sorted((DATA_DIR / 'raw').glob('dengue-cases-*.xlsx'))
if RAW_FILES:
    marikina_total = sum(len(pd.read_excel(f)) for f in RAW_FILES)
else:
    marikina_total = 6080   # value from 01_data_processing.ipynb (226+2114+1971+1515+254)
    print('Raw files not found — using the city total recorded in notebook 01.')

brgy_totals = df_real.groupby('barangay')['case_count'].sum()
city_share  = brgy_totals / marikina_total

print(f'Marikina cases 2022–2026: {marikina_total}')
for brgy in TARGET_BARANGAYS:
    print(f'  {brgy:<9} {brgy_totals[brgy]:>4} cases  →  {city_share[brgy]:.2%} of the city')
print(f'  Three barangays together: {brgy_totals.sum() / marikina_total:.2%} of the city')

# Seasonal pattern (share of a year's cases that fall in each week of the year)
df_real['week_of_year'] = df_real['week_start'].dt.isocalendar().week.astype(int)
seasonal = df_real.groupby('week_of_year')['case_count'].mean()
seasonal_weights = seasonal / seasonal.sum()

Marikina cases 2022–2026: 6080
  Nangka     381 cases  →  6.27% of the city
  Tumana     288 cases  →  4.74% of the city
  Malanday   319 cases  →  5.25% of the city
  Three barangays together: 16.25% of the city


## Step 2 — Generate synthetic weekly cases for 2017–2021

In [3]:
np.random.seed(42)

# DOH official yearly dengue totals for the whole of Marikina
doh_totals = {2017: 666, 2018: 1134, 2019: 1156, 2020: 273, 2021: 330}

synthetic_rows = []
for year, city_total in doh_totals.items():
    weeks = pd.date_range(start=f'{year}-01-01', periods=52, freq='W-MON')
    for brgy in TARGET_BARANGAYS:
        brgy_annual = city_total * city_share[brgy]          # corrected: share of the whole city
        for week_date in weeks:
            woy = week_date.isocalendar()[1]
            expected = brgy_annual * seasonal_weights.get(woy, 1 / 52)
            synthetic_rows.append({
                'barangay':   brgy,
                'week_start': week_date,
                'case_count': int(np.random.poisson(max(expected, 0))),
                'source':     'synthetic',
            })

df_synthetic = pd.DataFrame(synthetic_rows)

summary = df_synthetic.groupby(['barangay', df_synthetic['week_start'].dt.year])['case_count'].sum().unstack()
print('Synthetic cases per year (corrected):')
print(summary)
print('\nThree barangays vs whole city:')
for year, total in doh_totals.items():
    s = df_synthetic[df_synthetic['week_start'].dt.year == year]['case_count'].sum()
    print(f'  {year}: city {total:>5}  ·  three barangays {s:>4}  ({s / total:.1%})')

Synthetic cases per year (corrected):
week_start  2017  2018  2019  2020  2021
barangay                                
Malanday      40    64    51    19    18
Nangka        33    65    63    15    26
Tumana        30    60    63    18    13

Three barangays vs whole city:
  2017: city   666  ·  three barangays  103  (15.5%)
  2018: city  1134  ·  three barangays  189  (16.7%)
  2019: city  1156  ·  three barangays  177  (15.3%)
  2020: city   273  ·  three barangays   52  (19.0%)
  2021: city   330  ·  three barangays   57  (17.3%)


## Step 3 — Climate for 2017–2021, lagged 4 weeks like the recorded data

In [4]:
LAT, LON = 14.6507, 121.1029

def fetch_climate(start, end):
    url = ('https://archive-api.open-meteo.com/v1/archive'
           f'?latitude={LAT}&longitude={LON}&start_date={start}&end_date={end}'
           '&daily=precipitation_sum,temperature_2m_mean,relative_humidity_2m_mean'
           '&timezone=Asia/Manila')
    data = requests.get(url, timeout=60).json()
    return pd.DataFrame({
        'date':     pd.to_datetime(data['daily']['time']),
        'rainfall': data['daily']['precipitation_sum'],
        'temp':     data['daily']['temperature_2m_mean'],
        'humidity': data['daily']['relative_humidity_2m_mean'],
    })

# Start in November 2016 so the first weeks of 2017 have 4 earlier weeks to lag from
clim_parts = []
for start, end in [('2016-11-01', '2018-12-31'), ('2019-01-01', '2021-12-31')]:
    clim_parts.append(fetch_climate(start, end))
    time.sleep(1)
clim = pd.concat(clim_parts, ignore_index=True)

# Weekly totals/means, Monday-start weeks (same as notebook 01)
clim['week_start'] = clim['date'].dt.to_period('W').apply(lambda p: p.start_time)
clim_weekly = clim.groupby('week_start').agg(
    rainfall=('rainfall', 'sum'), temp=('temp', 'mean'), humidity=('humidity', 'mean')
).reset_index()

# 4-week lag: the value for week W is the climate of week W − 4
clim_lagged = clim_weekly.copy()
clim_lagged['week_start'] = clim_lagged['week_start'] + pd.Timedelta(weeks=4)
clim_lagged = clim_lagged.rename(columns={'rainfall': 'rainfall_lag',
                                          'temp': 'temp_lag',
                                          'humidity': 'humidity_lag'})

df_synthetic = df_synthetic.merge(clim_lagged, on='week_start', how='left')
print(f'Climate weeks: {len(clim_weekly)}')
print(f'Synthetic rows missing climate: {df_synthetic["rainfall_lag"].isna().sum()}')

Climate weeks: 270
Synthetic rows missing climate: 0


## Step 4 — Combine, label outbreaks, build features

In [5]:
real = df_real[['barangay', 'week_start', 'case_count',
                'rainfall_lag', 'temp_lag', 'humidity_lag']].copy()
real['source'] = 'real'

df_ext = (pd.concat([df_synthetic, real], ignore_index=True)
            .sort_values(['barangay', 'week_start'])
            .reset_index(drop=True))

# Outbreak = at or above the barangay's 75th percentile of RECORDED weekly cases
# (the threshold comes from real data only, so synthetic weeks can't move it)
p75 = real.groupby('barangay')['case_count'].quantile(0.75)
df_ext['outbreak'] = (df_ext['case_count'] >= df_ext['barangay'].map(p75)).astype(int)
print('Outbreak threshold (cases per week):', p75.to_dict())

df_ext.to_csv(DATA_DIR / 'processed_dataset_extended.csv', index=False)

le       = joblib.load(MODEL_DIR / 'label_encoder.pkl')
FEATURES = joblib.load(MODEL_DIR / 'feature_columns.pkl')

parts = []
for brgy in TARGET_BARANGAYS:
    d = df_ext[df_ext['barangay'] == brgy].sort_values('week_start').copy()
    for lag in [1, 2, 3, 4]:
        for col, name in [('rainfall_lag', 'rainfall'), ('temp_lag', 'temp'), ('humidity_lag', 'humidity')]:
            d[f'{name}_lag{lag}'] = d[col].shift(lag - 4) if lag < 4 else d[col]
    d['cases_roll4'] = d['case_count'].shift(1).rolling(4).mean()
    d['month'] = d['week_start'].dt.month
    d['prev_cases'] = d['case_count'].shift(1)          # for the "same as last week" baseline
    parts.append(d)

df_model = pd.concat(parts)
df_model['barangay_encoded'] = le.transform(df_model['barangay'])
df_model = df_model.dropna(subset=FEATURES).reset_index(drop=True)

# Train/test split by date: test = recorded weeks from TEST_START on
df_model['split'] = np.where((df_model['week_start'] >= TEST_START) & (df_model['source'] == 'real'),
                             'test', 'train')
train = df_model[df_model['split'] == 'train']
test  = df_model[df_model['split'] == 'test']

print(f'Training weeks: {len(train)}  (synthetic {int((train["source"] == "synthetic").sum())}, '
      f'recorded {int((train["source"] == "real").sum())})')
print(f'Test weeks:     {len(test)}  (all recorded, from {TEST_START.date()} — never seen in training)')

Outbreak threshold (cases per week): {'Malanday': 2.0, 'Nangka': 2.0, 'Tumana': 2.0}
Training weeks: 1230  (synthetic 768, recorded 462)
Test weeks:     186  (all recorded, from 2025-01-01 — never seen in training)


## Step 5 — Train on the training weeks, test on unseen recorded weeks

In [ ]:
def train_models(data):
    X, y_reg, y_clf = data[FEATURES], data['case_count'], data['outbreak']
    rf  = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y_reg)
    xg  = make_xgb().fit(X, y_reg)
    lr  = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000, random_state=42)).fit(X, y_clf)
    return rf, xg, lr

def regression_scores(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred   = np.asarray(pred, dtype=float)
    err    = np.abs(np.round(pred) - actual)
    return {'mae':     round(float(mean_absolute_error(actual, pred)), 2),
            'rmse':    round(float(np.sqrt(mean_squared_error(actual, pred))), 2),
            'within1': round(float((err <= 1).mean()) * 100, 1),
            'within2': round(float((err <= 2).mean()) * 100, 1)}

def classification_scores(actual, pred):
    return {'accuracy':  round(float(accuracy_score(actual, pred)) * 100, 1),
            'precision': round(float(precision_score(actual, pred, zero_division=0)) * 100, 1),
            'recall':    round(float(recall_score(actual, pred, zero_division=0)) * 100, 1),
            'f1':        round(float(f1_score(actual, pred, zero_division=0)) * 100, 1)}

rf_h, xgb_h, lr_h = train_models(train)

Xt = test[FEATURES]
rf_t  = np.maximum(0, rf_h.predict(Xt))
xgb_t = np.maximum(0, xgb_h.predict(Xt))
ens_t = 0.6 * rf_t + 0.4 * xgb_t
lr_t  = lr_h.predict(Xt)

# Baselines a forecasting model should beat
naive_last = test['prev_cases'].to_numpy(dtype=float)
naive_avg4 = test['cases_roll4'].to_numpy()
base_clf   = (naive_last >= test['barangay'].map(p75).to_numpy()).astype(int)

results = {
    'Random Forest':               regression_scores(test['case_count'], rf_t),
    'XGBoost':                     regression_scores(test['case_count'], xgb_t),
    'Ensemble (0.6 RF + 0.4 XGB)': regression_scores(test['case_count'], ens_t),
    'Baseline: same as last week': regression_scores(test['case_count'], naive_last),
    'Baseline: 4-week average':    regression_scores(test['case_count'], naive_avg4),
}
print('=== CASE COUNT — held-out recorded weeks ===')
print(pd.DataFrame(results).T.to_string())

clf_results = {
    'Logistic Regression':                classification_scores(test['outbreak'], lr_t),
    'Baseline: outbreak if last week was': classification_scores(test['outbreak'], base_clf),
}
print('\n=== OUTBREAK DETECTION — held-out recorded weeks ===')
print(pd.DataFrame(clf_results).T.to_string())

=== CASE COUNT — held-out recorded weeks ===
                              mae  rmse
Random Forest                1.16  1.59
XGBoost                      1.23  1.71
Ensemble (0.6 RF + 0.4 XGB)  1.17  1.61
Baseline: same as last week  1.38  1.92
Baseline: 4-week average     1.25  1.66

=== OUTBREAK DETECTION — held-out recorded weeks ===
                                     accuracy  precision  recall    f1
Logistic Regression                      69.9       77.6    45.8  57.6
Baseline: outbreak if last week was      65.6       61.2    62.7  61.9


In [ ]:
# Step 5b — Outbreak probability: Logistic Regression alone vs. blended with the case forecast
from scipy.stats import poisson

def poisson_outbreak_proba(expected_cases, thresholds):
    """Chance of reaching the outbreak threshold if weekly cases follow a Poisson count with this mean."""
    lam = np.maximum(np.asarray(expected_cases, dtype=float), 1e-6)
    k   = np.ceil(np.asarray(thresholds, dtype=float))
    return 1 - poisson.cdf(k - 1, lam)

def outbreak_proba(lr_model, rf_model, xgb_model, X, thresholds, method):
    p_lr = lr_model.predict_proba(X)[:, 1]
    if method == 'lr':
        return p_lr
    ens = 0.6 * np.maximum(0, rf_model.predict(X)) + 0.4 * np.maximum(0, xgb_model.predict(X))
    return 0.5 * p_lr + 0.5 * poisson_outbreak_proba(ens, thresholds)

# 1) Choose the method on 2024 only: train on weeks before 2024, validate on recorded 2024 weeks.
#    The 2025–2026 test weeks play no part in this choice.
VAL_START = pd.Timestamp('2024-01-01')
fit_v = train[train['week_start'] < VAL_START]
val   = train[(train['week_start'] >= VAL_START) & (train['source'] == 'real')]
rf_v, xgb_v, lr_v = train_models(fit_v)
thr_val = val['barangay'].map(p75).to_numpy()
val_scores = {}
for m in ['lr', 'blend']:
    pred = (outbreak_proba(lr_v, rf_v, xgb_v, val[FEATURES], thr_val, m) >= 0.5).astype(int)
    val_scores[m] = classification_scores(val['outbreak'], pred)
print('=== OUTBREAK METHOD — validation on 2024 (used to choose) ===')
print(pd.DataFrame(val_scores).T.to_string())
OUTBREAK_METHOD = 'blend' if val_scores['blend']['f1'] > val_scores['lr']['f1'] else 'lr'
print('Chosen outbreak method:', OUTBREAK_METHOD)

# 2) Score on the held-out 2025–2026 test (models trained before 2025, as in Step 5)
thr_test = test['barangay'].map(p75).to_numpy()
lr_only  = (outbreak_proba(lr_h, rf_h, xgb_h, Xt, thr_test, 'lr') >= 0.5).astype(int)
blend_t  = (outbreak_proba(lr_h, rf_h, xgb_h, Xt, thr_test, 'blend') >= 0.5).astype(int)
if OUTBREAK_METHOD == 'blend':
    clf_results = {
        'LR + Ensemble blend':                 classification_scores(test['outbreak'], blend_t),
        'Logistic Regression alone':           classification_scores(test['outbreak'], lr_only),
        'Baseline: outbreak if last week was': classification_scores(test['outbreak'], base_clf),
    }
else:
    clf_results = {
        'Logistic Regression':                 classification_scores(test['outbreak'], lr_only),
        'Baseline: outbreak if last week was': classification_scores(test['outbreak'], base_clf),
    }
print('\n=== OUTBREAK DETECTION — held-out recorded weeks ===')
print(pd.DataFrame(clf_results).T.to_string())

In [ ]:
# ── ACCURACY CHECK (read-only: changes no files, no models) ──
# Run after Step 5. Uses the same train/test split; the test weeks stay unseen.
from scipy.stats import poisson

y_true = test['case_count'].to_numpy()
preds = {
    'Random Forest': rf_t,
    'XGBoost':       xgb_t,
    'Ensemble':      ens_t,
    'Baseline: same as last week': naive_last,
    'Baseline: 4-week average':    naive_avg4,
}

# A. Case-count accuracy: share of test weeks where the forecast was within ±1 / ±2 cases
rows = {}
for name, p in preds.items():
    err = np.abs(np.round(p) - y_true)
    rows[name] = {'within ±1 case': round(float((err <= 1).mean()) * 100, 1),
                  'within ±2 cases': round(float((err <= 2).mean()) * 100, 1)}
print('=== A. CASE-COUNT ACCURACY (held-out test) ===')
print(pd.DataFrame(rows).T.to_string())

# B. Outbreak accuracy under different outbreak definitions.
# Thresholds come from RECORDED weeks BEFORE the test period only (no peeking at test data).
hist = df_model[(df_model['source'] == 'real') & (df_model['week_start'] < TEST_START)]
definitions = {
    'P75 (current)':        hist.groupby('barangay')['case_count'].quantile(0.75),
    'P90':                  hist.groupby('barangay')['case_count'].quantile(0.90),
    'Mean + 2 SD (epidemic)': hist.groupby('barangay')['case_count'].mean()
                              + 2 * hist.groupby('barangay')['case_count'].std(),
}

out = {}
for label, thr in definitions.items():
    t_train = (train['case_count'] >= train['barangay'].map(thr)).astype(int)
    t_test  = (test['case_count']  >= test['barangay'].map(thr)).astype(int)
    if t_train.nunique() < 2:
        print(f'{label}: no outbreak weeks in training — skipped'); continue
    lr_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)).fit(train[FEATURES], t_train)
    lr_pred = lr_b.predict(test[FEATURES])
    # Alert from the case forecast itself: chance of reaching the threshold, given the ensemble's expected cases
    thr_t   = test['barangay'].map(thr).to_numpy()
    p_reach = 1 - poisson.cdf(np.ceil(thr_t) - 1, np.maximum(ens_t, 1e-6))
    ens_pred = (p_reach >= 0.5).astype(int)
    always_no = np.zeros(len(t_test), dtype=int)
    for model, pr in [('Logistic Regression', lr_pred), ('Ensemble → Poisson', ens_pred), ('Always "no outbreak"', always_no)]:
        s = classification_scores(t_test, pr)
        out[(label, model)] = {**s, 'outbreak weeks in test': f'{t_test.mean() * 100:.0f}%'}
    print(f'{label}: threshold per barangay = {thr.round(1).to_dict()}')

print('\n=== B. OUTBREAK ACCURACY BY DEFINITION (held-out test) ===')
print(pd.DataFrame(out).T.to_string())

=== A. CASE-COUNT ACCURACY (held-out test) ===
                             within ±1 case  within ±2 cases
Random Forest                          71.0             89.2
XGBoost                                71.0             88.2
Ensemble                               72.0             89.8
Baseline: same as last week            61.3             81.7
Baseline: 4-week average               64.5             88.2
P75 (current): threshold per barangay = {'Malanday': 1.0, 'Nangka': 2.0, 'Tumana': 2.0}
P90: threshold per barangay = {'Malanday': 5.0, 'Nangka': 4.7, 'Tumana': 4.0}
Mean + 2 SD (epidemic): threshold per barangay = {'Malanday': 6.9, 'Nangka': 8.4, 'Tumana': 5.0}

=== B. OUTBREAK ACCURACY BY DEFINITION (held-out test) ===
                                            accuracy precision recall    f1 outbreak weeks in test
P75 (current)          Logistic Regression      65.6      74.6   47.3  57.9                    50%
                       Ensemble → Poisson       67.7      67.0   6

## Step 6 — Export `evaluation.json` for the web app
Every week gets predictions from the models trained **before 2025**, so test weeks are genuinely
out-of-sample. Headline metrics are computed on the **test weeks only**.

In [ ]:
X_all = df_model[FEATURES]
df_model['rf_pred']  = np.maximum(0, rf_h.predict(X_all))
df_model['xgb_pred'] = np.maximum(0, xgb_h.predict(X_all))
df_model['ens_pred'] = 0.6 * df_model['rf_pred'] + 0.4 * df_model['xgb_pred']
thr_all = df_model['barangay'].map(p75).to_numpy()
df_model['lr_pred']  = (outbreak_proba(lr_h, rf_h, xgb_h, X_all, thr_all, OUTBREAK_METHOD) >= 0.5).astype(int)

evaluation = {}
for brgy in TARGET_BARANGAYS:
    d  = df_model[df_model['barangay'] == brgy].sort_values('week_start')
    dt = d[d['split'] == 'test']
    reg_rf  = regression_scores(dt['case_count'], dt['rf_pred'])
    reg_xgb = regression_scores(dt['case_count'], dt['xgb_pred'])
    reg_ens = regression_scores(dt['case_count'], dt['ens_pred'])
    clf     = classification_scores(dt['outbreak'], dt['lr_pred'])
    tp = int(((dt['outbreak'] == 1) & (dt['lr_pred'] == 1)).sum())
    tn = int(((dt['outbreak'] == 0) & (dt['lr_pred'] == 0)).sum())
    fp = int(((dt['outbreak'] == 0) & (dt['lr_pred'] == 1)).sum())
    fn = int(((dt['outbreak'] == 1) & (dt['lr_pred'] == 0)).sum())
    evaluation[brgy] = {
        'weeks':      d['week_start'].dt.strftime('%Y-%m-%d').tolist(),
        'actual':     d['case_count'].astype(int).tolist(),
        'predicted':  d['rf_pred'].round().astype(int).tolist(),
        'xgb':        d['xgb_pred'].round().astype(int).tolist(),
        'ensemble':   d['ens_pred'].round().astype(int).tolist(),
        'source':     d['source'].tolist(),
        'split':      d['split'].tolist(),
        'clf_actual': d['outbreak'].astype(int).tolist(),
        'clf_pred':   d['lr_pred'].astype(int).tolist(),
        # headline numbers = held-out test weeks only
        'avg_error':          reg_rf['mae'],
        'xgb_avg_error':      reg_xgb['mae'],
        'ensemble_avg_error': reg_ens['mae'],
        'within1':     reg_rf['within1'],
        'within2':     reg_rf['within2'],
        'ens_within1': reg_ens['within1'],
        'ens_within2': reg_ens['within2'],
        'accuracy':  clf['accuracy'],
        'precision': clf['precision'],
        'recall':    clf['recall'],
        'f1':        clf['f1'],
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'total':      int(len(d)),
        'test_total': int(len(dt)),
        'test_start': TEST_START.strftime('%Y-%m-%d'),
    }

evaluation_meta = {
    'test_start':  TEST_START.strftime('%Y-%m-%d'),
    'train_weeks': int(len(train)),
    'test_weeks':  int(len(test)),
    'case_count':  results,
    'outbreak':    clf_results,
    'outbreak_method': OUTBREAK_METHOD,
}

with open(DATA_DIR / 'evaluation.json', 'w') as f:
    json.dump(evaluation, f)
with open(DATA_DIR / 'evaluation_summary.json', 'w') as f:
    json.dump(evaluation_meta, f, indent=2)

for brgy, r in evaluation.items():
    print(f'{brgy:<9} test weeks {r["test_total"]:>3} · RF ±{r["avg_error"]} · XGB ±{r["xgb_avg_error"]} '
          f'· Ens ±{r["ensemble_avg_error"]} · Acc {r["accuracy"]}% · Prec {r["precision"]}%')
print('\nSaved evaluation.json and evaluation_summary.json')

Nangka    test weeks  62 · RF ±1.25 · XGB ±1.3 · Ens ±1.23 · Acc 59.7% · Prec 71.4%
Tumana    test weeks  62 · RF ±0.87 · XGB ±0.96 · Ens ±0.89 · Acc 77.4% · Prec 82.4%
Malanday  test weeks  62 · RF ±1.33 · XGB ±1.45 · Ens ±1.37 · Acc 72.6% · Prec 77.8%

Saved evaluation.json and evaluation_summary.json


## Step 7 — Retrain on all data and save the models the app uses
The held-out test above measures how well the method works. The deployed models are then
trained on **all** weeks, so they learn from the most recent data too (standard practice).

In [ ]:
rf_f, xgb_f, lr_f = train_models(df_model)
joblib.dump(rf_f,  MODEL_DIR / 'rf_model.pkl')
joblib.dump(xgb_f, MODEL_DIR / 'xgb_model.pkl')
joblib.dump(lr_f,  MODEL_DIR / 'lr_model.pkl')
joblib.dump({'method': OUTBREAK_METHOD,
             'thresholds': {b: float(v) for b, v in p75.items()}},
            MODEL_DIR / 'outbreak_config.pkl')
print('Saved rf_model.pkl, xgb_model.pkl, lr_model.pkl (trained on all weeks)')
print(f'Saved outbreak_config.pkl · method = {OUTBREAK_METHOD}')

Saved rf_model.pkl, xgb_model.pkl, lr_model.pkl (trained on all weeks)


## Step 8 — Regenerate `heatmap.json` with the corrected synthetic data

In [9]:
heatmap = {'barangays': {}}
for brgy in TARGET_BARANGAYS:
    d = df_ext[df_ext['barangay'] == brgy].sort_values('week_start')
    weeks = []
    for _, row in d.iterrows():
        c = int(row['case_count'])
        risk = 'none' if c == 0 else 'low' if c <= 2 else 'moderate' if c <= 5 else 'high'
        weeks.append({'week': row['week_start'].strftime('%Y-%m-%d'),
                      'cases': c, 'risk': risk, 'source': row['source']})
    heatmap['barangays'][brgy] = weeks
heatmap['max_cases'] = max(w['cases'] for v in heatmap['barangays'].values() for w in v)

with open(DATA_DIR / 'heatmap.json', 'w') as f:
    json.dump(heatmap, f)
print(f'Saved heatmap.json · max {heatmap["max_cases"]} cases in a week')

Saved heatmap.json · max 28 cases in a week
